### Import required libraries

In [1]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer

### Download NLTK resources

In [2]:
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

### Load your scraped dataset

In [3]:
df = pd.read_csv("../data/imdb_movies_2024_6000.csv")

df.head()

,Movie_Title,Storyline
0,Arthur the King,An adventure racer adopts a stray dog named Ar...
1,Deadpool & Wolverine,Deadpool is offered a place in the Marvel Cine...
2,Gladiator II,After his home is conquered by the tyrannical ...
3,Anora,A young stripper from Brooklyn impulsively mar...
4,The Ministry of Ungentlemanly Warfare,The British military recruits a small group of...


In [4]:
# Check shape:
print("Original shape:", df.shape)

Original shape: (5946, 2)


### Remove missing storylines

In [5]:
df = df.dropna(subset=["Storyline"]).copy()

df.reset_index(drop=True, inplace=True)

print("Shape after removing missing storylines:", df.shape)

Shape after removing missing storylines: (5731, 2)


In [6]:
### Check for empty storylines
empty_storylines = (
    df["Storyline"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("Empty storylines:", empty_storylines)

Empty storylines: 0


In [7]:
df = df[
    df["Storyline"].astype(str).str.strip() != ""
].copy()

df.reset_index(drop=True, inplace=True)

In [8]:
# Check duplicate rows
print("Total duplicate rows:", df.duplicated().sum())

# Check duplicate movie titles
print("Duplicate movie titles:", df["Movie_Title"].duplicated().sum())

# Remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)

Total duplicate rows: 0
Duplicate movie titles: 0
Shape after removing duplicates: (5731, 2)


### Cleaning the storylines

In [9]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove special characters and numbers
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenize
    words = text.split()

    # Remove stopwords and lemmatize
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)


df["Cleaned_Storyline"] = df["Storyline"].apply(clean_text)

df[["Movie_Title", "Storyline", "Cleaned_Storyline"]].head()

,Movie_Title,Storyline,Cleaned_Storyline
0,Arthur the King,An adventure racer adopts a stray dog named Ar...,adventure racer adopts stray dog named arthur ...
1,Deadpool & Wolverine,Deadpool is offered a place in the Marvel Cine...,deadpool offered place marvel cinematic univer...
2,Gladiator II,After his home is conquered by the tyrannical ...,home conquered tyrannical emperor lead rome lu...
3,Anora,A young stripper from Brooklyn impulsively mar...,young stripper brooklyn impulsively marries so...
4,The Ministry of Ungentlemanly Warfare,The British military recruits a small group of...,british military recruit small group highly sk...


In [10]:
# Check for empty cleaned storylines
empty_cleaned = (df["Cleaned_Storyline"].str.strip() == "").sum()

print("Empty cleaned storylines:", empty_cleaned)
print("Dataset shape:", df.shape)

Empty cleaned storylines: 0
Dataset shape: (5731, 3)


In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf.fit_transform(df["Cleaned_Storyline"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of movies:", tfidf_matrix.shape[0])
print("Number of features:", tfidf_matrix.shape[1])

TF-IDF matrix shape: (5731, 10000)
Number of movies: 5731
Number of features: 10000


In [28]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_movies(user_storyline, top_n=5):
    # Clean user input
    cleaned_input = clean_text(user_storyline)

    # Convert user input to TF-IDF vector
    input_vector = tfidf.transform([cleaned_input])

    # Calculate cosine similarity with all movie storylines
    similarity_scores = cosine_similarity(
        input_vector,
        tfidf_matrix
    ).flatten()

    # Get indices of top similar movies
    top_indices = similarity_scores.argsort()[::-1][:top_n]

    # Create result dataframe
    recommendations = df.iloc[top_indices][
        ["Movie_Title", "Storyline"]
    ].copy()

    recommendations["Similarity_Score"] = similarity_scores[top_indices]

    return recommendations

In [29]:
story = """
A young wizard joins a magical school, makes new friends,
learns magic and fights against dark forces.
"""

recommend_movies(story)

,Movie_Title,Storyline,Similarity_Score
296,Witchboard,"A cursed Witchboard awakens dark forces, dragg...",0.233217
361,Orion and the Dark,A boy with an active imagination faces his fea...,0.230072
2314,Deaner '89,A hilarious headbanger finally makes it after ...,0.223791
1970,Panda Bear in Africa,"A panda travels from China to Africa, facing h...",0.215082
1375,Super Vixens 7,Ancient God empowers Skeletor against She-Ra. ...,0.196675
